# ROSP-11: Time-loss accounting (weather / fault / scheduled)

PTH Tier 1 metric **ROSP-11** - {{ params.start_dayobs }} to {{ params.end_dayobs }} (dayObs, inclusive)

- Jira epic: [ROSP-11](https://rubinobs.atlassian.net/browse/ROSP-11); loss-cause taxonomy: [ROSP-24](https://rubinobs.atlassian.net/browse/ROSP-24)
- Confluence (Hub page; no metric page yet): [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696)
- Performance Tracking Hub: [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696)

| | |
|---|---|
| Steward | Brian Stalder |
| Owner | Sandrine Thomas |
| Category | 1 - Observatory Operations |
| Layer | 3 |

Method version: `rosp-11-v1.0` **draft**, built on loss-cause taxonomy **v1.0 (pending ROSP-24 sign-off)** with the rule updates decided on 2026-11-16/17 (Decision Log v32; ROSP-24, ROSP-42, ROSP-58, ROSP-23, ROSP-11). Until the taxonomy is signed and the method tag is cut, no number on this page is a baselined value. Changes to the method go through the PTH Method Change Process: https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/2032959501

**Method.** Every second of the observable night (Sun at -12° in the evening to -12° in the morning, [ROSP-22](https://rubinobs.atlassian.net/browse/ROSP-22); weather is *not* excluded) is assigned to exactly one ROSP-24 category. Where several apply, the first in the precedence order wins:

**B1 → C → B2 → D (D2, D1, D4, D3) → E → A → F**

| | Category | Evidence used here |
|---|---|---|
| A1 | Survey open-shutter time | ConsDB exposures of the survey programs, `img_type = science` |
| A2 | Overhead within the slew model | gaps between consecutive survey exposures up to the `rubin_scheduler` slew model (via `rubin_nights`) plus a scatter allowance; other survey-program exposures |
| A3 | Start-up within the allowance | BLOCK-T539 (initial alignment, always the start-up) and idle time inside the 10-min allowance from the start-up origin (-12° now; dome release if the ROSP-58 Q6 amendment is accepted), up to the first survey science visit |
| B1 | Planned maintenance | ROSP-42 override file, `category = B1` |
| B2a / B2b | On-sky / closed-dome engineering and calibration (good conditions) | non-survey, non-start-up LSSTCam exposures, **only while no `FAULT` or `DOWNTIME` state is active**; dome shutter open / closed |
| C1 | Closed by conditions | `WEATHER` state, or measured weather (tower limits of the Weather Constraints version in force, allskycam index > 0.5, rain), with the dome closed |
| C2 | Open but unusable | allskycam index > 0.5, dome open, no survey activity |
| C3 | Weather standby | after a weather closure: until the re-opening criteria of the Weather Constraints version in force are met; post-precipitation / icing inspection; and `WEATHER` state or limits with the dome open and no survey activity |
| C4 / C5 (and the storm C1) | Summit access closed / neighbouring observatories | ROSP-42 override file. C5 applies only where there is no EFD weather data |
| D1 | Short fault | `FAULT` state; a start-up overrun (including T539 outside the allowance) |
| D2 | Long fault | a night with no survey science visit whose `FAULT` state covers the whole night |
| D3 | Unreported fault | the part of a survey visit gap beyond the slew-model allowance, for gaps up to the D3 cap |
| D4 | Unplanned downtime | unplanned `DOWNTIME` state; override `D4` (with measured weather carved out as C); override C5 where EFD weather data shows no weather; a closed dome in good conditions with nothing running and no weather reason |
| E | Operational idle | `IDLE`, or `OPERATIONAL` with no survey activity; visit gaps longer than the D3 cap with no `FAULT` and no engineering. Reported with D1 for now |
| F | Unattributed | `UNKNOWN` state, no fresh state |

The Scheduler observatory states are applied first and never edited. Where the shared override file has a row, it replaces the recorded state (not the weather record) inside its window ([ROSP-42](https://rubinobs.atlassian.net/browse/ROSP-42)); the file is read from `main` at run time and its commit SHA is recorded in the provenance box. The reader also accepts the proposed `group` column, windows inside a dayObs and the categories D1 / D2 / B2a (ROSP-58 decision 2), none of which the file on `main` uses yet. Engineering vs survey comes from program metadata (`science_program`), not from the states. Weather during maintenance and faults under weather are computed as a **shadow record**, shown, and never added to the ledger. Fault hours go to a primary subsystem group: the first CSC to enter `FAULT`, from the Scheduler status notes as parsed in `fault_downtime.ipynb`, checked against the CSC summary states. A `FAULT` with no CSC named (typically set by hand) goes to F (unattributed); the 24 h look-back is kept only as `suggested_group`. Corrections that the narrative log, OBS and FRACAS support (ROSP-58) are shown as a what-if table and not applied: the override file is the single place they will be applied.

The rules that are still parameters pending sign-off are listed in the *Rules* table below. The shared functions live in [`rosp11_ledger.py`](rosp11_ledger.py) in this folder.

In [ ]:
# Parameters cell. Set defaults here - gets replaced by the sidecar on Times Square.
# Defaults are the PTH baseline epoch.
start_dayobs = 20260629  # First dayObs (inclusive)
end_dayobs = (
    20261231  # Last dayObs (inclusive); nights that have not started are skipped
)

not_times_square = True

In [ ]:
# The version of rubin_nights that defines this metric (twilight times, observatory states, slew model). Pinned as in
# ROSP-9 so an upstream change cannot silently change the series; changing it is a method change (record it in
# METHOD_CHANGES at the end).
RUBIN_NIGHTS_TAG = "v0.18.2"

# Times Square runs this notebook in a Nublado pod where rubin_nights may be missing or at another version; install the
# pinned tag there. Anywhere else (e.g. an SDF login node) this cell is a no-op.
try:
    not_times_square
except NameError:
    not_times_square = False

if not not_times_square:
    print(f"installing rubin_nights {RUBIN_NIGHTS_TAG}")
    !pip install --user --upgrade git+https://github.com/lsst-sims/rubin_nights.git@{RUBIN_NIGHTS_TAG} --no-deps > /dev/null 2>&1

In [ ]:
# Imports, the shared ledger module, and the connections. Imports live here (not in the parameters cell) because
# Times Square replaces the parameters cell wholesale.
import dataclasses
import datetime
import os
import platform
import shutil
import subprocess
import sys
import types
import urllib.request
import warnings
from importlib.metadata import version as package_version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.time import Time, TimeDelta
from IPython.display import HTML, display

from rubin_nights import connections
import rubin_nights.dayobs_utils as rn_dayobs

%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
NOTEBOOK_DIR = os.getcwd()

# Bumped through the PTH Method Change Process; the baseline commit will be tagged rosp-11-vX.Y.
METHOD_VERSION = "rosp-11-v1.0 (draft, rule updates 2026-11-16/17)"

start_dayobs = rn_dayobs.day_obs_str_to_int(str(start_dayobs))
end_dayobs = rn_dayobs.day_obs_str_to_int(str(end_dayobs))
if start_dayobs > end_dayobs:
    raise ValueError(
        f"start_dayobs ({start_dayobs}) must be <= end_dayobs ({end_dayobs})"
    )

rubin_nights_version = package_version("rubin_nights")
if rubin_nights_version != RUBIN_NIGHTS_TAG.lstrip("v"):
    warnings.warn(
        f"rubin_nights {rubin_nights_version} is loaded but this notebook is pinned to {RUBIN_NIGHTS_TAG}; "
        "the results may not match the recorded series."
    )
print(f"rubin_nights {rubin_nights_version}")

# The shared functions are in rosp11_ledger.py next to this notebook. Times Square executes only the notebook (no
# checkout), so when the file is not beside the notebook it is fetched from GitHub; its git blob SHA goes in the
# provenance box either way.
LEDGER_FILE = "rosp11_ledger.py"
LEDGER_REPO_PATH = "tracking_hub/ops/rosp-11_time_loss_accounting/" + LEDGER_FILE
LEDGER_REF = os.environ.get("ROSP11_LEDGER_REF", "main")
# DRAFT ONLY: until this notebook is merged the module is not on main yet. Remove before merging.
LEDGER_FALLBACK_REFS = ("tickets/ROSP-11",)
EXPECTED_LEDGER_API = 2


def load_ledger_module():
    local = os.path.join(NOTEBOOK_DIR, LEDGER_FILE)
    if os.path.exists(local):
        with open(local, "rb") as f:
            data, origin = f.read(), local
    else:
        data, errors = None, []
        for ref in (LEDGER_REF,) + LEDGER_FALLBACK_REFS:
            url = f"https://raw.githubusercontent.com/lsst/sp_notebooks/{ref}/{LEDGER_REPO_PATH}"
            try:
                with urllib.request.urlopen(url, timeout=30) as response:
                    data, origin = response.read(), url
                break
            except Exception as err:
                errors.append(f"{url}: {err}")
        if data is None:
            raise RuntimeError(
                "could not load " + LEDGER_FILE + ": " + "; ".join(errors)
            )
    module = types.ModuleType("rosp11_ledger")
    module.__file__ = origin
    sys.modules["rosp11_ledger"] = module  # dataclasses look the module up here
    exec(compile(data, origin, "exec"), module.__dict__)
    return module, origin, data


ledger, ledger_origin, _ledger_bytes = load_ledger_module()
ledger_blob_sha = ledger.git_blob_sha(_ledger_bytes)
if ledger.MODULE_API_VERSION != EXPECTED_LEDGER_API:
    raise RuntimeError(
        f"{LEDGER_FILE} API {ledger.MODULE_API_VERSION} does not match this notebook ({EXPECTED_LEDGER_API})"
    )
print(f"ledger module: {ledger_origin} (blob {ledger_blob_sha[:12]})")

# The Sun-position code needs the shared rubin_sim data. If it is not set, point at the shared copy at USDF.
if os.getenv("RUBIN_SIM_DATA_DIR") is None:
    for _data_dir in (
        "/sdf/data/rubin/shared/rubin_sim_data",
        "/sdf/group/rubin/shared/rubin_sim_data",
    ):
        if os.path.isdir(_data_dir):
            os.environ["RUBIN_SIM_DATA_DIR"] = _data_dir
            break

# On an RSP (including Times Square) the defaults are right. Elsewhere get_clients() looks for your own USDF token.
endpoints = connections.get_clients()
efd_client, consdb_client = endpoints["efd"], endpoints["consdb"]

# Figure style (dataviz reference palette; categorical slots 1-5 in their validated order, grey for F).
INK, INK_MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e1e0d9", "#fcfcfb"
GROUP_COLORS = {
    "A": "#2a78d6",
    "B": "#eb6834",
    "C": "#1baf7a",
    "D": "#eda100",
    "E": "#e87ba4",
    "F": "#898781",
}
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "legend.frameon": False,
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.titlelocation": "left",
    }
)

## Data sources

| Input | Source | Used for |
|---|---|---|
| Observable night | `rubin_nights.dayobs_utils.day_obs_sunset_sunrise` at -12° | the denominator; every hour of it is assigned |
| Observatory state | EFD `lsst.sal.Scheduler.logevent_observatoryStatus` (salIndex 1), via `rubin_nights.observatory_status` | `WEATHER`, `FAULT`, `DOWNTIME`, `IDLE`, `OPERATIONAL`, `UNKNOWN` |
| State overrides | `tracking_hub/ops/observatory_state_overrides/observatory_state_overrides.csv` on `main`, pinned to the last commit that touched it (`ROSP_STATE_OVERRIDES_CSV` overrides the location) | whole-night B1 / C1 / C4 / C5 / D4 where the EFD record is missing or wrong |
| Fault CSCs | the "are in `<State.FAULT>`" clause of the status notes (as `fault_downtime.ipynb`), checked against EFD `logevent_summaryState` | primary subsystem group |
| Exposures | ConsDB `cdb_lsstcam.exposure` (all image types) | A1, A2, D3 (survey programs) and B2 (everything else) |
| Slew model | `rubin_nights.rubin_scheduler_addons.add_model_slew_times` (TMA limits from the EFD) | the A2 / D3 split of each visit gap |
| Weather tower | EFD `lsst.sal.ESS.airFlow`, `relativeHumidity`, `dewPoint`, `temperature` (salIndex 301), 1-min bins | C1 closing limits and C3 re-opening criteria per Weather Constraints version; icing; "good conditions" for B2b and D4 |
| Rain | EFD `lsst.sal.ESS.logevent_precipitation` `raining` (salIndex 301) | C1, and the post-precipitation C3 hold |
| Weather Constraints | dated versions in `Rules.weather_constraints` (Confluence page 39696225); **placeholder dates and re-opening values** until confirmed | which limits and re-opening criteria apply on each night |
| Dome shutter | EFD `lsst.sal.MTDome.apertureShutter` `positionActual0/1`, 1-min max | open vs closed (C1 / C2 / C3, B2a / B2b, D4) |
| Allskycam | `allsky_cloud_frames.parquet` under `SYSTEM_PERFORMANCE_DATA_DIR` (SDF only; reduction in bstalder/rubin_allskycam) | cloud index for C1 / C2 / C3. **Optional**: where it is missing, cloud enters only through the `WEATHER` state and C2 is not measured; those nights are flagged *cloud-provisional* |

The narrative log, night reports and FRACAS are annotation sources under ROSP-24 and do not set hours here.

### Rules

Every rule that ROSP-24 has not yet fixed to a number, and every number this notebook had to choose, is a field of `ledger.Rules`. The table is what this run used; `pending` rows are the ones still waiting on a decision. The second table is the dated Weather Constraints versions: **every date and re-opening value in it is a placeholder** until the page history is supplied (the 2026-10-01 wind switch is assumed, and the 2026-11-06 row marks the version ROSP-58 calls "11/06 adopted").

In [ ]:
rules = ledger.Rules()
with pd.option_context("display.max_colwidth", None):
    display(rules.as_table())
    display(rules.weather_table())

## Selection

The nights are every dayObs from `start_dayobs` to `end_dayobs` whose observable night has started; a night in progress is counted up to now. Nothing is removed: weather, faults and maintenance are categories of the ledger, not cuts.

In [ ]:
today_dayobs = rn_dayobs.day_obs_str_to_int(rn_dayobs.today_day_obs())
nights = ledger.night_windows(
    ledger.day_obs_range(start_dayobs, min(end_dayobs, today_dayobs)),
    rules.twilight_alt,
)
if len(nights) == 0:
    raise ValueError(
        f"no observable night in {start_dayobs} to {end_dayobs} has started yet"
    )
first_night, last_night = int(nights.index.min()), int(nights.index.max())
t_start = rn_dayobs.day_obs_to_time(first_night)
t_end = rn_dayobs.day_obs_to_time(last_night) + TimeDelta(1, format="jd")
print(
    f"{len(nights)} nights, dayObs {first_night} to {last_night}, {nights.night_hours.sum():.1f} h of observable night"
    + (
        ""
        if nights.complete.all()
        else f" (night {last_night} in progress, counted to now)"
    )
)

# The shared override file, read from main at the commit that last changed it.
overrides, overrides_prov = ledger.read_overrides(
    os.environ.get("ROSP_STATE_OVERRIDES_CSV")
)
if overrides_prov["inconsistent_rows"]:
    warnings.warn(
        f"override rows whose category disagrees with states/planned_downtime: {overrides_prov['inconsistent_rows']}"
    )
print(
    f"overrides: {overrides_prov['n_rows']} rows {overrides_prov['first']} to {overrides_prov['last']}, "
    f"commit {overrides_prov['commit_sha']}, blob {overrides_prov['blob_sha'][:12]}"
)
if overrides_prov.get("error"):
    warnings.warn(
        f"override commit SHA not resolved, read the branch tip instead: {overrides_prov['error']}"
    )
in_range = overrides[overrides.day_obs.isin(nights.index)]
print(
    f"{len(in_range)} override rows in range: "
    + (
        ", ".join(f"{k} {v}" for k, v in in_range.category.value_counts().items())
        or "none"
    )
)
print(
    f"group column populated: {overrides_prov['has_group_column']}; partial-night windows: {overrides_prov['partial_windows']}"
)
override_night_category = (
    overrides[overrides.whole_night].set_index("day_obs").category.to_dict()
)

In [ ]:
# Observatory states, the fault runs and their primary CSC.
status_messages = ledger.fetch_status_messages(efd_client, t_start, t_end)
state_iv = ledger.state_intervals(status_messages, nights, rules.max_state_age_h)
state_iv, fault_runs = ledger.fault_runs(
    state_iv, status_messages, rules.fault_onset_lookback_h
)
onset_cscs = sorted({c for names in fault_runs.onset_cscs for c in names})
summary_states = ledger.fetch_summary_states(
    efd_client, onset_cscs, t_start - TimeDelta(3, format="jd"), t_end
)
fault_runs = ledger.assign_primary_csc(fault_runs, summary_states, rules)
fresh_hours = state_iv.loc[state_iv.status >= 0, "hours"].sum()
print(
    f"{len(status_messages)} status messages; fresh state on {fresh_hours:.1f} of {nights.night_hours.sum():.1f} h "
    f"(override nights included); {len(fault_runs)} fault runs"
)
print(fault_runs.primary_rule.value_counts().to_string())

In [ ]:
# Environment, allskycam, exposures and the slew model.
env = ledger.fetch_environment(efd_client, first_night, last_night)
print(f"weather tower / dome: {len(env)} one-minute bins")

ALLSKYCAM_FRAMES = os.environ.get(
    "ROSP11_ALLSKYCAM_FRAMES",
    os.path.join(
        os.environ.get(
            "SYSTEM_PERFORMANCE_DATA_DIR",
            "/sdf/data/rubin/user/stalder/system_performance_data",
        ),
        "allsky_cloud_frames.parquet",
    ),
)
allskycam, allskycam_note, allskycam_last = ledger.load_allskycam(
    ALLSKYCAM_FRAMES, first_night, last_night
)
print(f"allskycam: {allskycam_note}")
rain, n_rain = ledger.fetch_precipitation(
    efd_client, t_start - TimeDelta(1, format="jd"), t_end, rules.rain_stale_h
)
print(f"rain: {n_rain} 'raining' reports, {len(rain)} rain intervals")
weather_evidence = ledger.weather_minutes(env, allskycam, rain, nights, rules)
print(
    f"weather evidence: {len(weather_evidence)} night minutes; versions used: {sorted(set(weather_evidence.version))}"
)

exposures = ledger.fetch_exposures(consdb_client, first_night, last_night)
survey = exposures[exposures.science_program.isin(rules.survey_programs)]
survey, slew_status = ledger.add_slew_model(survey, efd_client)
exposures = exposures.merge(
    survey[["exposure_id", "slew_model"]], on="exposure_id", how="left"
)
sequence = ledger.survey_sequence(exposures, rules)
print(
    f"{len(exposures)} exposures, {len(survey)} in the survey programs; slew model: {slew_status}"
)

## Computation

The night is cut into elementary segments at every boundary of every input (state messages, exposure starts and ends, the edges of each gap allowance, minute bins of the weather tower, dome and allskycam, the end of the start-up allowance). Each segment gets the evidence in force at its midpoint, and the first applicable category in the precedence order. Two passes follow:

- **Start-up.** BLOCK-T539 (initial alignment) is always start-up, never engineering. The allowance runs for `startup_allowance_min` from the start-up origin (`startup_origin`: -12° now; with `dome_release`, from each dome opening after a closure, and -12° → first release is D4). T539 time inside the allowance is A3, and so is idle (E/F) time before the first survey science visit. T539 time outside the allowance, and the idle stretch from the end of the allowance to the first survey visit (stopping at the first segment of another category), is the start-up overrun, charged as `startup_overrun_category` (D1).
- **Whole-night fault.** A night with no survey science visit whose `FAULT` state covers the whole night to within `d2_tolerance_min` is D2: its time not already claimed by B1 or C becomes D2 (B2 cannot apply under a fault).

The rule updates of 2026-11-16/17 change four more things. **B2 needs an available system**: engineering or calibration while a `FAULT` or `DOWNTIME` state is active is D, and the activity goes in the shadow record. **Measured weather beats override D4** (C comes before D): inside an override D4 window, weather measured by the tower, the allskycam or the rain gauge is C, and C5 applies only where there is no EFD weather data (elsewhere the measurement decides: C where weather is measured, D4 where not). **C3 weather standby**: after a weather closure (closing conditions with the dome shut), the time until the re-opening criteria of the Weather Constraints version in force have held for `reopen_hold_min`, the `precip_hold_min` after rain, and after icing the time until the dome next opens; a closed dome in good conditions with no weather reason stays D4. **D3 cap**: a survey visit gap longer than the cap (`d3_cap_s`; by default the 99th percentile of the non-fault survey-to-survey gaps in the range, ROSP-23) with no `FAULT` and no engineering inside is E, not D3.

Where the definitions overlap by construction, this notebook reads them as follows (each is listed in the PR as a point for the sign-off): E and A3 exclude survey activity, so `IDLE` during an exposure is A1; A3 takes E/F time inside the allowance even though E precedes A; the `WEATHER` state during survey activity leaves the time in A, because C1 is "closed" and C2/C3 require no science; a closed dome without dome telemetry is treated as closed for C1.

The per-night hours must add up to the night length exactly; the cell asserts it.

In [ ]:
d3_cap_s = (
    rules.d3_cap_s
    if rules.d3_cap_s is not None
    else ledger.d3_cap_default(sequence["gaps"], state_iv, rules.d3_cap_quantile)
)
ledger_inputs = dict(
    windows=nights,
    intervals=state_iv,
    runs=fault_runs,
    overrides=overrides,
    env=env,
    cam=allskycam,
    wxm=weather_evidence,
    seq=sequence,
    rules=rules,
    d3_cap_s=d3_cap_s,
)
segments = ledger.build_ledger(**ledger_inputs)
per_night = ledger.per_night_table(segments, nights)
assert per_night.closure_s.abs().max() < 1e-3, per_night.closure_s.abs().max()
print(
    f"{len(segments)} segments; every night closes to {per_night.closure_s.abs().max():.1e} s; "
    f"D3 cap {d3_cap_s:.0f} s ({'set' if rules.d3_cap_s is not None else f'{rules.d3_cap_quantile:.0%} of non-fault survey gaps'})"
)
cat_summary, group_summary, short_summary, total_hours = ledger.cycle_summary(per_night)

# Cloud measurement per night: nights after the allskycam record ends are cloud-provisional.
cam_nights = set(allskycam.day_obs) if allskycam is not None else set()
cloud_status = pd.Series(
    [
        (
            "allskycam"
            if d in cam_nights
            else (
                "not available here"
                if allskycam is None
                else (
                    f"provisional: allskycam ends {allskycam_last}"
                    if allskycam_last is not None and d > allskycam_last
                    else "no allskycam data"
                )
            )
        )
        for d in nights.index
    ],
    index=nights.index,
)
dome_coverage = (
    segments.assign(h=np.where(segments.dome_open.notna(), segments.hours, 0.0))
    .groupby("day_obs")
    .h.sum()
    / nights.night_hours
)

## Results

### Per-night ledger

Hours by category for every night in the range (columns in category order; `night_hours` is the -12° to -12° night). `cloud` says whether the allskycam measured the night; nights after the allskycam record ends are **cloud-provisional** (cloud enters only through the `WEATHER` state until the processing is extended). `dome data` is the fraction of the night with dome telemetry. The figure shows the same nights by group, and below it where in the night each group fell.

In [ ]:
cols = [c for c in ledger.CATEGORY_ORDER if per_night[c].sum() > 0.005]
table = per_night[cols + ["night_hours"]].copy()
table.insert(0, "override", [override_night_category.get(d, "") for d in table.index])
table["cloud"] = cloud_status.reindex(table.index)
table["dome data"] = dome_coverage.reindex(table.index).round(2)
with pd.option_context("display.max_rows", None):
    display(table.round(2))

In [ ]:
def by_group(frame):
    return pd.DataFrame(
        {
            g: frame[[c for c in ledger.CATEGORY_ORDER if c[0] == g]].sum(axis=1)
            for g in ledger.GROUP_NAMES
        }
    )


nightly_groups = by_group(per_night)
dates = pd.to_datetime(nightly_groups.index.astype(str), format="%Y%m%d")
fig, axes = plt.subplots(
    2, 1, figsize=(15, 8.5), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]}
)
ax = axes[0]
bottom = np.zeros(len(nightly_groups))
width = 0.8 if len(nightly_groups) <= 60 else 1.0
for g, name in ledger.GROUP_NAMES.items():
    h = nightly_groups[g].values
    if h.sum() <= 0:
        continue
    ax.bar(
        dates,
        h,
        bottom=bottom,
        width=width,
        color=GROUP_COLORS[g],
        edgecolor=SURFACE,
        linewidth=0.8,
        label=f"{g} {name} ({h.sum():.0f} h)",
    )
    bottom += h
ax.set_ylabel("hours of observable night")
ax.set_title("Where each night went, by ROSP-24 group")
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.08), ncol=3, fontsize=9)

# Within-night timeline: consecutive segments of the same group merged into one block.
ax = axes[1]
seg = segments.assign(group=segments.category.str[0])
seg["run"] = (
    (seg.group != seg.group.shift()) | (seg.day_obs != seg.day_obs.shift())
).cumsum()
blocks = seg.groupby("run").agg(
    day_obs=("day_obs", "first"),
    group=("group", "first"),
    start=("start", "min"),
    end=("end", "max"),
)
night_start = nights.start.reindex(blocks.day_obs).values
x = pd.to_datetime(blocks.day_obs.astype(str), format="%Y%m%d")
for g in ledger.GROUP_NAMES:
    sel = (blocks.group == g).values
    if sel.any():
        ax.bar(
            x[sel],
            ((blocks.end - blocks.start).values[sel]) * 24,
            bottom=(blocks.start.values[sel] - night_start[sel]) * 24,
            width=width,
            color=GROUP_COLORS[g],
            linewidth=0,
        )
ax.set_ylabel("hours after evening -12°")
ax.set_ylim(nights.night_hours.max() * 1.02, 0)
ax.set_title("When in the night (top = evening twilight)")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### Cycle fractions of the observable night

Hours and fraction of the observable night by category, by group, and the short losses: E is its own category but is reported together with D1 for now (Q7), so D1, E and their sum are shown.

In [ ]:
display(cat_summary.style.format({"hours": "{:.2f}", "fraction": "{:.1%}"}))
display(group_summary.style.format({"hours": "{:.2f}", "fraction": "{:.1%}"}))
display(short_summary.style.format({"hours": "{:.2f}", "fraction": "{:.1%}"}))

fig, ax = plt.subplots(figsize=(9, 6.5))
shown = cat_summary.iloc[::-1]
y = np.arange(len(shown))
ax.barh(
    y, shown.fraction * 100, color=[GROUP_COLORS[c[0]] for c in shown.index], height=0.7
)
for yi, (c, row) in zip(y, shown.iterrows()):
    ax.text(
        row.fraction * 100,
        yi,
        f"  {row.fraction:.1%}  ({row.hours:.1f} h)",
        va="center",
        fontsize=8.5,
        color=INK,
    )
ax.set_yticks(y)
ax.set_yticklabels([f"{c}  {ledger.CATEGORIES[c]}" for c in shown.index], fontsize=8.5)
ax.set_xlabel("% of observable night")
ax.set_xlim(0, max(shown.fraction.max() * 100 * 1.35, 1))
ax.set_title(
    f"dayObs {first_night} to {last_night}: {total_hours:.0f} h of observable night ({len(nights)} nights)"
)
plt.tight_layout()
plt.show()

### Fault loss by subsystem group

D1, D2 and D4 hours by ROSP-24 attribution group. `FAULT`-state hours go to the group of the run's primary CSC: the CSC in the first current `FAULT` clause of the run, or of the fault stretch it continues (the `primary_rule` column of the run table says which source was used, and when several CSCs are named the CSC summary states decide which entered `FAULT` first). A `FAULT` with no current clause naming a CSC (typically set by hand) goes to **F (unattributed)**; the run table's `suggested_group` keeps the 24 h look-back (or the sticky "were in" clause) as a suggestion only. D2 hours without a fault run of their own take the night's largest run. D4 for a closed dome with nothing running is "Operations decision"; override D4 takes the override row's `group` (none yet: "Pending attribution"); other D4 and the start-up overrun have no subsystem in the data and stay "Pending attribution". D3 is not attributed. The last two columns compare this with the equal split of each run over every CSC it names that `fault_downtime.ipynb` used before #39.

In [ ]:
faults = ledger.fault_by_group(segments, fault_runs)
display(faults.round(2))

fig, ax = plt.subplots(figsize=(9, 0.45 * len(faults) + 1.2))
y = np.arange(len(faults))[::-1]
ax.barh(y, faults.total.values, color=GROUP_COLORS["D"], height=0.6)
for yi, h in zip(y, faults.total.values):
    ax.text(h, yi, f"  {h:.1f} h", va="center", fontsize=9, color=INK)
ax.set_yticks(y)
ax.set_yticklabels(faults.index, fontsize=9)
ax.set_xlabel("D1 + D2 + D4 hours")
ax.set_xlim(0, max(faults.total.max() * 1.25, 1))
ax.set_title("Fault loss by subsystem group")
plt.tight_layout()
plt.show()

# The fault runs behind the table, largest first.
seg_fault = segments[segments.category.isin(["D1", "D2"]) & segments.fault_run.notna()]
charged = seg_fault.groupby("fault_run").hours.sum().rename("charged_h")
run_table = fault_runs.set_index("fault_run").join(charged).fillna({"charged_h": 0.0})
run_table = run_table.sort_values("charged_h", ascending=False)
with pd.option_context("display.max_colwidth", 90):
    display(
        run_table[
            [
                "day_obs",
                "hours",
                "charged_h",
                "primary_csc",
                "group",
                "suggested_group",
                "onset_source",
                "all_cscs",
                "first_note",
            ]
        ]
        .head(20)
        .round(2)
    )

### Pending decisions: sensitivity

The same ledger with the rules that still wait on a decision set otherwise: the D3 cap (ROSP-23; Erik to set it), the start-up origin (ROSP-58 Q6 amendment), and whether a `FAULT` with no CSC moves to category F rather than to group F. Hours by category; only the affected categories are shown.

In [ ]:
variants = {
    f"D3 cap {d3_cap_s:.0f} s (default)": (rules, d3_cap_s),
    "D3 cap 300 s": (rules, 300.0),
    "D3 cap 1800 s": (rules, 1800.0),
    "no D3 cap (previous rule)": (rules, None),
    "start-up from dome release": (
        dataclasses.replace(rules, startup_origin="dome_release"),
        d3_cap_s,
    ),
    "FAULT with no CSC -> category F": (
        dataclasses.replace(rules, no_csc_fault_category="F"),
        d3_cap_s,
    ),
}
sens = ledger.sensitivity(ledger_inputs, variants, nights)
sens = sens.loc[(sens.max(axis=1) - sens.min(axis=1)) > 0.005]
display(sens.round(2))

### Pending attribution corrections (what-if, not applied)

Corrections that the narrative log, OBS and FRACAS support but that are not yet rows in the override file (ROSP-58). Each is applied alone to a copy of the ledger and the hours it would move are shown; the last row applies all non-conditional ones together. None of this is in the ledger above: the override file is the single place a correction is applied. The ROSP-58 start-up rows need windows inside a night and the `group` column, which the file does not have yet; where the dome was already open at twilight, the release is taken as the first T539 or survey exposure. The ROSP-25 quality measure below counts these corrections against the automatic attributions.

In [ ]:
def night_bounds(d):
    return (nights.loc[d].start, nights.loc[d].end) if d in nights.index else None


def first_startup(d):
    e = exposures[
        (exposures.day_obs == d)
        & exposures.science_program.isin(rules.survey_programs + rules.startup_programs)
    ]
    return e.obs_start_mjd.min() if len(e) else np.nan


corrections, out_of_range = [], []


def add(c):
    (
        corrections
        if c.get("day_obs") is None or c["day_obs"] in nights.index
        else out_of_range
    ).append(c)


add(
    dict(
        id="20261101 -> rotator",
        kind="fault_group",
        day_obs=20261101,
        group="Telescope mount and structure",
        evidence="FRACAS-474; operator note (rotator); fault message names MTCamera",
        status="evidenced; needs the group column",
    )
)
add(
    dict(
        id="20261115 -> camera",
        kind="fault_group",
        day_obs=20261115,
        group="Camera",
        evidence="FAULT set by hand, no CSC; camera fault ongoing (ROSP-58)",
        status="pending row (ROSP-58)",
    )
)
add(
    dict(
        id="20260707 -> camera (conditional)",
        kind="fault_group",
        day_obs=20260707,
        group="Camera",
        conditional=True,
        evidence="operator note: camera team recovering; fault message names MTMount",
        status="only if an OBS/FRACAS ticket confirms",
    )
)
mirror = fault_runs[
    (fault_runs.day_obs == 20260630)
    & fault_runs.first_note.fillna("").str.contains("intentionally faulting")
]
for _, m in mirror.iterrows():
    add(
        dict(
            id="20260630 deliberate mirror fault -> B2a",
            kind="window",
            day_obs=20260630,
            start=m.start,
            end=m.end,
            category="B2a",
            evidence="status note: 'intentionally faulting the mirror ... Not an actual fault'",
            status="pending row; Q8 stays the rule",
        )
    )
if (overrides.category == "D4").any() and overrides.day_obs.isin(nights.index).any():
    add(
        dict(
            id="override D4 with no group -> site/external",
            kind="override_group",
            category="D4",
            group="Site and external",
            evidence="storm-recovery D4 (ROSP-58 proposal)",
            status="proposed, pending Erik / Sandrine",
        )
    )
if night_bounds(20261028):
    s, e = night_bounds(20261028)
    add(
        dict(
            id="ROSP-58 20261028 -> D4, measured cloud stays C",
            kind="window",
            day_obs=20261028,
            start=s,
            end=e,
            category="D4",
            group="Operations decision",
            evidence="recorded WEATHER all night; limits misapplied",
            status="pending row; group may be control software (Watcher alarm)",
        )
    )
for d, g in (
    (20261107, "Facility utilities"),
    (20261108, "Facility utilities"),
    (20261109, ledger.PENDING),
):
    if night_bounds(d):
        rel = first_startup(d)
        if np.isfinite(rel) and rel > nights.loc[d].start:
            add(
                dict(
                    id=f"ROSP-58 {d}: -12 deg -> release is D4 ({g})",
                    kind="window",
                    day_obs=d,
                    start=nights.loc[d].start,
                    end=rel,
                    category="D4",
                    group=g,
                    evidence=f"start-up began {ledger.tai_mjd_to_utc_iso(rel)[11:16]} UTC",
                    status="pending row; needs partial-night windows",
                )
            )
        else:
            add(
                dict(
                    id=f"ROSP-58 {d}: -12 deg -> release is D4 ({g})",
                    kind="note",
                    day_obs=d,
                    status="no delay in the data",
                    evidence="start-up exposures began at or before -12 deg",
                    moves="nothing to move: check the dayObs",
                )
            )
if night_bounds(20261110):
    add(
        dict(
            id="ROSP-58 20261110 mount fault -> mount",
            kind="fault_group",
            day_obs=20261110,
            group="Telescope mount and structure",
            evidence="ROSP-58; fault message names MTHexapod:1",
            status="pending; confirm the first CSC into FAULT",
        )
    )
if night_bounds(20261111):
    t0 = Time("2026-11-12T02:30:00", scale="utc").tai.mjd
    a1 = segments[
        (segments.day_obs == 20261111)
        & (segments.start >= t0)
        & (segments.category == "A1")
    ].hours.sum()
    add(
        dict(
            id="ROSP-58 20261111 D4 (dome; operations decision from 02:30 UTC)",
            kind="note",
            day_obs=20261111,
            evidence="ROSP-58",
            status="not applied: dayObs to confirm",
            moves=f"the window holds {a1:.2f} h of survey open-shutter time",
        )
    )
if out_of_range:
    print("not in this range:", ", ".join(c["id"] for c in out_of_range))
if corrections:
    what_if, segments_corrected = ledger.what_if_table(
        ledger_inputs, segments, corrections
    )
    with pd.option_context("display.max_colwidth", None):
        display(what_if)

# ROSP-25 attribution quality: automatic attributions and the corrections pending against them.
state_fault = segments[
    segments.category.isin(["D1", "D2"]) & segments.fault_run.notna()
]
auto = state_fault[state_fault.fault_group != ledger.UNATTRIBUTED_FAULT]
fault_corrections = [
    c
    for c in corrections
    if c["kind"] == "fault_group"
    or (c["kind"] == "window" and c.get("category") == "B2a")
]
corrected_h = sum(
    (
        state_fault[state_fault.day_obs == c["day_obs"]].hours.sum()
        if c["kind"] == "fault_group"
        else state_fault[
            (state_fault.start >= c["start"]) & (state_fault.end <= c["end"])
        ].hours.sum()
    )
    for c in fault_corrections
    if not c.get("conditional")
)
n_runs = state_fault.fault_run.nunique()
print(
    f"ROSP-25: {n_runs} fault runs charged ({state_fault.hours.sum():.1f} h of FAULT-state D1/D2); attributed automatically "
    f"{auto.fault_run.nunique()} runs, {auto.hours.sum():.1f} h ({auto.hours.sum() / max(state_fault.hours.sum(), 1e-9):.0%}); "
    f"unattributed (group F) {state_fault.hours.sum() - auto.hours.sum():.1f} h. Attribution corrections pending: "
    f"{sum(1 for c in fault_corrections if not c.get('conditional'))} (+{sum(1 for c in fault_corrections if c.get('conditional'))} conditional), "
    f"touching {corrected_h:.1f} h ({corrected_h / max(state_fault.hours.sum(), 1e-9):.0%} of the FAULT-state hours)."
)

### Shadow record

Time where a lower-precedence category also applied, kept for statistics (weather during maintenance for the year-over-year weather record; faults under weather for ROSP-12 and the simulation validation; engineering or calibration run while a `FAULT` or `DOWNTIME` state was active, which is charged to D). **None of it is in the ledger totals above.** Measured weather is the weather-tower limits, the allskycam index or the `WEATHER` state, whichever is available.

In [ ]:
display(ledger.shadow_table(segments).round(2))
shadow_nights = ledger.shadow_by_night(segments)
if len(shadow_nights):
    with pd.option_context("display.max_rows", None):
        display(shadow_nights.round(2))
else:
    print("no shadow-record time in this range")

### Checks for review

Nights with unattributed (F) time, the D2 nights, the input coverage per night, and the places where the precedence moved time away from what the observing record shows. These are not part of the ledger; they are what the sign-off review should look at.

In [ ]:
f_nights = segments[segments.category == "F"].pivot_table(
    index="day_obs", columns="detail", values="hours", aggfunc="sum", fill_value=0.0
)
if len(f_nights):
    f_nights["F total"] = f_nights.sum(axis=1)
    print(f"Nights with F time ({len(f_nights)}):")
    display(f_nights.round(3))
else:
    print("No F (unattributed) time in this range.")

d2 = segments.attrs.get("d2_nights", [])
print(f"\nD2 (whole-night fault) nights: {d2 or 'none'}")
if d2:
    display(
        segments[segments.category == "D2"]
        .groupby(["day_obs", "fault_group", "fault_csc"])
        .hours.sum()
        .round(2)
        .to_frame()
    )

coverage = (
    segments.assign(
        state=np.where(segments.status >= 0, segments.hours, 0.0),
        weather_tower=np.where(segments.wx_bad.notna(), segments.hours, 0.0),
        dome=np.where(segments.dome_open.notna(), segments.hours, 0.0),
        allskycam=np.where(segments.cloudy.notna(), segments.hours, 0.0),
    )
    .groupby("day_obs")[["state", "weather_tower", "dome", "allskycam"]]
    .sum()
    .div(nights.night_hours, axis=0)
)
coverage["override"] = [override_night_category.get(d, "") for d in coverage.index]
print("\nInput coverage (fraction of each night; override nights need no EFD state):")
print(
    coverage.drop(columns="override")
    .describe()
    .loc[["mean", "min"]]
    .round(2)
    .to_string()
)
incomplete = coverage[
    (coverage.override == "")
    & (coverage[["state", "weather_tower", "dome"]].min(axis=1) < 0.95)
]
if len(incomplete):
    print(
        f"\nNon-override nights with < 95 % state, weather-tower or dome coverage ({len(incomplete)}):"
    )
    display(incomplete.round(2))

provisional = cloud_status[cloud_status.str.startswith("provisional")]
print(
    f"\nCloud-provisional nights ({len(provisional)}): {list(provisional.index) or 'none'}"
)
no_dome = dome_coverage[
    (dome_coverage < 0.05) & ~dome_coverage.index.isin(list(override_night_category))
]
for d in no_dome.index:
    print(
        f"No dome telemetry on {d}: "
        + ", ".join(
            f"{k} {v:.2f} h"
            for k, v in per_night.loc[d, ledger.CATEGORY_ORDER].items()
            if v > 0.005
        )
    )
print("\nWhere the precedence moved time (for review):")
with pd.option_context("display.max_colwidth", None):
    display(ledger.precedence_checks(segments).round(2))
gaps = sequence["gaps"]
if len(gaps):
    resid = (gaps.gap_s - gaps.model_s).dropna()
    print(
        f"visit gaps: {len(gaps)}; gap - slew model, median {resid.median():.1f} s, 90th pct {resid.quantile(0.9):.1f} s, "
        f"95th {resid.quantile(0.95):.1f} s; {(gaps.excess_s > 0).mean():.1%} have D3 excess at gap_scatter_s = "
        f"{rules.gap_scatter_s:g} s; D3 from gaps > 30 min: {gaps.loc[gaps.gap_s > 1800, 'excess_s'].sum() / 3600:.2f} h "
        f"of {gaps.excess_s.sum() / 3600:.2f} h"
    )

## Provenance

What this run was computed with. The override file is read at the commit shown, so the ledger can be reproduced from it; the ledger module is identified by its git blob SHA (`git hash-object rosp11_ledger.py`).

In [ ]:
def stack_version():
    if shutil.which("eups"):
        res = subprocess.run(
            ["eups", "list", "-s", "lsst_distrib"], capture_output=True, text=True
        )
        return "lsst_distrib " + (" ".join(res.stdout.split()) or "not set up")
    return os.environ.get("IMAGE_DESCRIPTION", "eups not available")


def local_git_commit(path):
    try:
        folder = os.path.dirname(path)
        sha = subprocess.run(
            [
                "git",
                "-C",
                folder,
                "log",
                "-1",
                "--format=%H",
                "--",
                os.path.basename(path),
            ],
            capture_output=True,
            text=True,
        ).stdout.strip()
        dirty = subprocess.run(
            [
                "git",
                "-C",
                folder,
                "status",
                "--porcelain",
                "--",
                os.path.basename(path),
            ],
            capture_output=True,
            text=True,
        ).stdout.strip()
        return (sha or "not committed") + (" + local changes" if dirty else "")
    except Exception:
        return "n/a"


cam_cover = (
    segments.loc[segments.cloudy.notna(), "hours"].sum() / total_hours
    if total_hours
    else 0.0
)
provenance = {
    "Method version": METHOD_VERSION,
    "Taxonomy version": rules.taxonomy_version,
    "Run (UTC)": datetime.datetime.now(datetime.timezone.utc).strftime(
        "%Y-%m-%d %H:%M"
    ),
    "dayObs range": f"{first_night} to {last_night} ({len(nights)} nights, {total_hours:.1f} h)",
    "Stack": stack_version(),
    "rubin_nights": f"{rubin_nights_version} (pinned {RUBIN_NIGHTS_TAG})",
    "Override file": overrides_prov["url"],
    "Override commit SHA": overrides_prov["commit_sha"],
    "Override blob SHA": overrides_prov["blob_sha"],
    "Ledger module": f"{ledger_origin} (blob {ledger_blob_sha}"
    + (
        f", commit {local_git_commit(ledger_origin)})"
        if os.path.exists(ledger_origin)
        else ")"
    ),
    "Slew model": slew_status,
    "Allskycam": f"{allskycam_note}; covers {cam_cover:.0%} of the observable night",
    "Weather Constraints": "; ".join(
        f"{r['version']} from {r['from_dayobs']} ({r['status']})"
        for r in rules.weather_constraints
    ),
    "D3 cap": f"{d3_cap_s:.0f} s",
    "Start-up origin": rules.startup_origin,
    "Kernel image": os.environ.get("JUPYTER_IMAGE_SPEC", "not a Nublado pod"),
    "python / numpy / pandas": f"{platform.python_version()} / {np.__version__} / {pd.__version__}",
}
rows = "".join(
    f"<tr><td style='padding:2px 14px 2px 0; vertical-align:top;'>{k}</td><td><code>{v}</code></td></tr>"
    for k, v in provenance.items()
)
display(
    HTML(
        "<div style='border:2px solid #52514e; border-radius:6px; padding:10px 16px; max-width:1100px; font-family:sans-serif;'>"
        "<div style='font-weight:bold; letter-spacing:0.04em; margin-bottom:6px;'>ROSP-11 PROVENANCE</div>"
        f"<table style='border-collapse:collapse; font-size:0.9em;'>{rows}</table></div>"
    )
)

## Configuration-change markers

Changes that can move the series for reasons other than system performance. By the PTH rules a change of definition is a **series break**, not a restatement: record it here with its dayObs. `CONFIG_CHANGES` (system changes) stays empty until the stewards agree what to record.

In [ ]:
METHOD_CHANGES = {
    20260629: (
        "Method rosp-11-v1.0 (draft) on taxonomy v1.0 (pending ROSP-24 sign-off), rubin_nights "
        f"{RUBIN_NIGHTS_TAG}. Any change to Rules, the override file or the pinned library is a method change."
    ),
    20260715: (
        "From here to 20261025 the night is set by observatory_state_overrides.csv (storm C1, Sep C4/C5/D4, "
        "B1 maintenance 0914-1004, Oct C4/C5/D4); the EFD state is not used on those nights."
    ),
    20260719: "Allskycam down 20260719 to 20261009: cloud enters only through the WEATHER state; C2 not measured.",
    20261001: "Weather Constraints WC-2: wind limits 15/18 -> 20/25 m/s (ROSP-24 C1; switch date unverified, placeholder).",
    20261027: "Survey start: science_program BLOCK-407 for every survey visit.",
    20261103: "Last allskycam night processed: later nights are cloud-provisional until the processing is extended.",
    20261106: "Weather Constraints WC-3 ('11/06 adopted version', ROSP-58; values are placeholders).",
    20261117: (
        "Rule updates of 2026-11-16/17 (Decision Log v32) applied from the start of the series: B2 only while available; "
        "T539 is start-up; measured weather beats override D4 and C5 only without EFD weather data; C3 weather standby; "
        "D3 cap; a FAULT with no CSC goes to F."
    ),
}
CONFIG_CHANGES = {}

for title, changes in (
    ("Method changes", METHOD_CHANGES),
    ("System changes", CONFIG_CHANGES),
):
    in_range = {d: txt for d, txt in changes.items() if d <= last_night}
    print(f"{title}:")
    for d, txt in sorted(in_range.items()):
        print(f"  {d}: {txt}")
    if not in_range:
        print("  none recorded for this range")